In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [2]:
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
import matplotlib
import scipy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib.colors as mcolors

from datetime import datetime, timedelta
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import netCDF4
import string

import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
from datetime import timedelta
import gsw
import ArcTools as Atools
import pandas as pd
import xarray as xr
import numpy as np
import pandas as pd
from matplotlib.path import Path
from scipy.stats import linregress
import gc


In [3]:


def detrend_temp_anomalies_per_basin(
    input_file,
    box_file,
    output_file,
    anomaly_var="temperature_QC_anom",
    detrended_var="temperature_QC_anom_dtd",
    p_threshold=0.05,
):
    """
    Detrend temperature anomalies inside spatial boxes.

    Parameters
    ----------
    input_file : str
        Path to anomaly NetCDF file.
    box_file : str
        CSV file containing spatial box definitions.
    output_file : str
        Output NetCDF filename.
    anomaly_var : str
        Name of anomaly variable.
    detrended_var : str
        Name of detrended anomaly variable.
    p_threshold : float
        Significance threshold for detrending.
    """

    print(f"\nProcessing: {input_file}")

    # --------------------------------------------------
    # Load data
    # --------------------------------------------------
    ds_anom = xr.open_dataset(input_file, decode_times=False)

    boxes = pd.read_csv(box_file)

    # --------------------------------------------------
    # Build polygons
    # --------------------------------------------------
    xc = np.array([
        boxes["Longitude 1"].values,
        boxes["Longitude 2"].values,
        boxes["Longitude 2"].values,
        boxes["Longitude 1"].values,
        boxes["Longitude 1"].values
    ]).T

    yc = np.array([
        boxes["Latitude 1"].values,
        boxes["Latitude 1"].values,
        boxes["Latitude 2"].values,
        boxes["Latitude 2"].values,
        boxes["Latitude 1"].values
    ]).T

    box_id = np.array(boxes["Box ID"].values)

    # --------------------------------------------------
    # Find profiles inside each box
    # --------------------------------------------------
    profiles_in_box = {}

    points = np.column_stack((
        ds_anom.longitude.values,
        ds_anom.latitude.values
    ))

    for i in range(len(xc)):
        

        polygon = np.column_stack((xc[i], yc[i]))
        path = Path(polygon)

        inside = path.contains_points(points)

        profiles_in_box[box_id[i]] = np.where(inside)[0]

        nprof_box = len(profiles_in_box[box_id[i]])

        if nprof_box > 0:
            print(f"Box {box_id[i]}: {nprof_box} profiles inside")

    # --------------------------------------------------
    # Prepare arrays
    # --------------------------------------------------
    nbox = len(box_id)
    ndepth = ds_anom.depth.size

    slope_box_depth = np.full((nbox, ndepth), np.nan)
    pval_box_depth = np.full((nbox, ndepth), np.nan)

    temp_anom_detrended = np.full(
        ds_anom[anomaly_var].shape,
        np.nan
    )

    temp_anom_array = np.array(ds_anom[anomaly_var])
    time_array = np.array(ds_anom.time)

   # --------------------------------------------------
    # Detrending loop
    # --------------------------------------------------
    for i in range(nbox):

        
        current_box = box_id[i]
        prof_idx = profiles_in_box[current_box]

        # Skip if too few profiles
        if len(prof_idx) < 4:
            temp_anom_detrended[prof_idx,:] = temp_anom_array[prof_idx,:]
            continue
        
        print(f"Detrending box: {current_box}")
        time = time_array[prof_idx]

        
        t = time

        for j in range(ndepth):

            temp_anom = temp_anom_array[prof_idx, j]

            mask = np.isfinite(temp_anom)

            if (np.sum(mask) > 4) and ((np.max(t[mask]) -np.min(t[mask]))/365.25>5): # at least 5 years

                slope, intercept, rvalue, pvalue, _ = linregress(
                    t[mask],
                    temp_anom[mask]
                )

                slope_box_depth[i, j] = slope
                pval_box_depth[i, j] = pvalue

                trend = slope * t + intercept

                # Remove trend only if significant
                if pvalue < p_threshold:
                    temp_anom_detrended[prof_idx, j] = (
                        temp_anom - trend
                    )
                else:
                    temp_anom_detrended[prof_idx, j] = temp_anom

            else:
                temp_anom_detrended[prof_idx, j] = temp_anom

    # --------------------------------------------------
    # Save detrended field
    # --------------------------------------------------
    ds_anom[detrended_var] = (
        ["profile", "depth"],
        temp_anom_detrended
    )

#     # Optional: save regression diagnostics
#     ds_anom["trend_slope"] = (
#         ["box", "depth"],
#         slope_box_depth
#     )

#     ds_anom["trend_pvalue"] = (
#         ["box", "depth"],
#         pval_box_depth
#     )

#     ds_anom["box"] = box_id

    # ds_anom.to_netcdf(output_file)

    print(f"Saved: {output_file}")
  



In [4]:

box_file = (
    "/home/aprigent/Documents/Projects/Analysis_obs/"
    "Arctic_design_10m_150_test_new.csv"
)


basins = [
    "barents",
    "eurasian",
    "amerasian",
    "nordic",
    "amerasian_shelf",
    "siberian_shelf",
    "baffin",
    "kara"
]

for basin in basins:

    input_file = (
        f"/data0/user/aprigent/PROCESSED/level_4/"
        f"temp_anomalies_relative_ISAS_{basin}_full_after_2005.nc"
    )

    output_file = (
        f"/data0/user/aprigent/PROCESSED/level_5/"
        f"temp_anomalies_detrended_relative_ISAS_{basin}_full_after_2005.nc"
    )

    detrend_temp_anomalies_per_basin(
        input_file=input_file,
        box_file=box_file,
        output_file=output_file,
        anomaly_var="temperature_QC_anom",
        detrended_var="temperature_QC_anom_dtd",
        p_threshold=0.05,
    )


Processing: /data0/user/aprigent/PROCESSED/level_4/temp_anomalies_relative_ISAS_barents_full_after_2005.nc
Box 90: 1 profiles inside
Box 91: 3 profiles inside
Box 92: 1 profiles inside
Box 131: 73 profiles inside
Box 132: 499 profiles inside
Box 133: 4565 profiles inside
Box 134: 1741 profiles inside
Box 136: 7 profiles inside
Box 137: 7 profiles inside
Box 138: 4 profiles inside
Box 183: 74 profiles inside
Box 184: 1912 profiles inside
Box 185: 3303 profiles inside
Box 186: 1811 profiles inside
Box 187: 746 profiles inside
Box 188: 1123 profiles inside
Box 189: 289 profiles inside
Box 190: 214 profiles inside
Box 191: 100 profiles inside
Box 192: 35 profiles inside
Box 193: 7 profiles inside
Box 242: 759 profiles inside
Box 243: 3410 profiles inside
Box 244: 1157 profiles inside
Box 245: 934 profiles inside
Box 246: 1613 profiles inside
Box 247: 428 profiles inside
Box 248: 115 profiles inside
Box 249: 108 profiles inside
Box 250: 34 profiles inside
Box 298: 740 profiles inside
Box 2